<div style="background-color: #f5f5f5; padding: 20px; border-radius: 10px; border: 1px solid #e0e0e0;">
<table style="width:100%; border:none;">
  <tbody><tr style="border:none;">
    <td style="width:20%; text-align:center; border:none; vertical-align:middle;">
      <img src="./UI-Logo.png" alt="UI Logo" width="100">
    </td>
    <td style="width:60%; text-align:center; border:none; vertical-align:middle;">
      <h1 style="margin:0; color:#333;">University of Isfahan</h1>
      <h2 style="margin:0; color:#555; font-weight:normal;">Faculty of Mathematics and Statistics</h2>
      <br/>
      <hr style="border: 1px solid #ccc; margin-top:15px; margin-bottom:15px;">
      <h3 style="margin:0; color:#0056b3;">Outlier Detection</h3>
      <br/>
      <h4 style="margin:0; font-size: 1.1em; color: #333; line-height: 1.6; margin-top: 15px; display: inline-block;">
        <strong>Course:</strong> Medical Data Mining<br>
        <strong>Instructor:</strong> Dr. H. Marateb<br>
        <strong>Semester:</strong> Winter 1404-1405
      </h4>
    </td>
    <td style="width:20%; text-align:center; border:none; vertical-align:middle;">
      <img src="./UI-Logo.png" alt="UI Logo" width="100">
    </td>
  </tr>
</tbody></table>
</div>

<div style="background-color: #f5f5f5; border: 1px solid #f5f5f5; padding: 20px; border-radius: 10px; margin-top: 25px; margin-bottom: 25px;">
  
  <h3 style="text-align: center; margin-top: 0; color: #0056b3;">Student Information</h3>

  <table style="width: 60%; margin-left: auto; margin-right: auto; border: none; font-size: 1.1em; color: #333;">
    <tbody>
      <tr>
        <td style="text-align: right; padding: 8px; width: 35%;"><strong>Student:</strong></td>
        <td style="text-align: left; padding: 8px;"><em>Zahra Cheraghi Seydani</em></td>
      </tr>
      <tr>
        <td style="text-align: right; padding: 8px; width: 35%;"><strong>Student:</strong></td>
        <td style="text-align: left; padding: 8px;"><em>Mohammad Mallaee</em></td>
      </tr>
      <tr>
        <td style="text-align: right; padding: 8px; width: 35%;"><strong>Student:</strong></td>
        <td style="text-align: left; padding: 8px;"><em>Davoud Nosrati Amirabadi</em></td>
      </tr>
    </tbody>
  </table>
  
</div>

## Import the necessary libraries

In [ ]:
import pandas as pd
import numpy as np

## Load Excel file

In [ ]:
pd.set_option({"display.float_format":"{:.5f}".format})
e = pd.read_excel("./data.xlsx")
data = e[['Variable_1', "Variable_2", "Variable_3"]]
data

## 3 Sigma

In [ ]:
mean = data.mean()
std = data.std()

three_sigma_up = mean + 3*std
three_sigma_low = mean - 3*std

print("-"*10, "MEAN", "-"*10)
print(mean, end="\n\n\n")

print("-"*10, "STD", "-"*10)
print(std, end="\n\n\n")

print("-"*10, "3 Sigma boundaries", "-"*10)
for var in data.columns:
    print(f"{var:12} {three_sigma_low[var]:8.5f} to {three_sigma_up[var]:9.5f}")
print("\n")

outliers_mask = (data < three_sigma_low) | (data > three_sigma_up)
data[outliers_mask].dropna(how="all")

As we can observe, Variable_1 has 3 outliers:


*3 values, not shown: the dataset is under an NDA.*

They are indeed, out of the specified boundary for Variable_1:
(34.98501, 66.57349)

Secondly, Variable_2 also has 3 outliers:

*3 values, not shown: the dataset is under an NDA.*

which are clearly outside (72.18563, 131.12670).

Lastly, there are 3 outliers for Variable_3 as well:

*3 values, not shown: the dataset is under an NDA.*

They are also out of (-0.31661, 2.05691).

## Hampel

In [ ]:
median = data.median()
abs_dev = np.abs(data - median)
mad = 1.6 * abs_dev.median()

hampel_low = median - 5*mad
hampel_up = median + 5*mad

print("-"*10, "MEDIAN", "-"*10)
print(median, end="\n\n\n")

print("-"*10, "MAD", "-"*10)
print(mad, end="\n\n\n")


print("-"*10, "Hampel boundaries", "-"*10)
for col in data.columns:
    print(f"{col:12} {hampel_low[col]:8.5f} to {hampel_up[col]:9.5f}")
print("\n")

outliers_mask = (hampel_low > data) | (data > hampel_up)
data[outliers_mask].dropna(how="all")


Let's see the results:

Variable_1 -- (30.61000, 70.77000):

*2 values, not shown: the dataset is under an NDA.*

Variable_2 -- (63.53000, 140.01000):

*2 values, not shown: the dataset is under an NDA.*

Variable_3 -- (-0.37050, 2.18950):

*3 values, not shown: the dataset is under an NDA.*


It's obviously less strict than 3-sigma; in fact, it has not detected rows 26 and 56 as outliers.

## Tukey

In [ ]:
Q1 = pd.Series(np.percentile(data, 25, method="weibull", axis=0), index=data.columns, name="Q1")
Q3 = pd.Series(np.percentile(data, 75, method="weibull", axis=0), index=data.columns, name="Q3")

# Q1 = data.quantile(0.25, axis=0)
# Q3 = data.quantile(0.75, axis=0)
IQR = Q3 - Q1

tukey_low = Q1 - 1.5*IQR
tukey_up = Q3 + 1.5*IQR


print("-"*10, "Q1 Quartile", "-"*10)
print(Q1, end='\n\n\n')

print("-"*10, "Q3 Quartile", "-"*10)
print(Q3, end='\n\n\n')

print("-"*10, "IQR Quartile", "-"*10)
print(IQR, end='\n\n\n')

print("-"*10, "Tukey Boundaries", "-"*10)
for col in data.columns:
    print(f"{col:12} {tukey_low[col]:8.5f} to {tukey_up[col]:9.5f}")

print("\n")

outliers_mask = (tukey_up < data) | (data < tukey_low)
data[outliers_mask].dropna(how="all")

All the outliers mentioned by the 2 previous methods are included, as well as 2 new ones. So it seems Tukey is a stricter method for outlier detection as its interval is shorter:

Variable_1 -- (40.44125, 60.73125):

*4 values, not shown: the dataset is under an NDA.*

Variable_2 -- (81.59750, 121.05750):

*3 values, not shown: the dataset is under an NDA.*

Variable_3 -- (0.22950, 1.54150):

*4 values, not shown: the dataset is under an NDA.*
